In [2]:
import os
import librosa
import numpy as np
import random

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.regularizers import l2

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [3]:
asthma_data = "./Datasets/Asthma"
copd_data = "./Datasets/COPD4"
healthy_data = "./Datasets/Healthy"

print("Asthma files:", len(os.listdir(asthma_data)))
print("COPD files:", len(os.listdir(copd_data)))
print("Healthy files:", len(os.listdir(healthy_data)))

Asthma files: 96
COPD files: 112
Healthy files: 112


In [4]:
def augment_noise(audio, sr=None):
    noise = np.random.randn(len(audio))
    return audio + 0.005 * noise

def augment_pitch(audio, sr):
    return librosa.effects.pitch_shift(audio, sr=sr, n_steps=random.uniform(-2, 2))

def augment_speed(audio, sr):
    speed = random.uniform(0.9, 1.1)
    return librosa.effects.time_stretch(audio, rate=speed)

def apply_augmentations(audio, sr=22050):
    funcs = [
        lambda y: augment_noise(y),
        lambda y: augment_pitch(y, sr),
        lambda y: augment_speed(y, sr)
    ]
    func = random.choice(funcs)
    return func(audio)

In [5]:
def extract_mfcc(audio, sr=22050, max_len=130):
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=40,
        n_fft=2048,
        hop_length=512
    )

    mfcc = mfcc.T 

    if mfcc.shape[0] < max_len:
        pad_width = max_len - mfcc.shape[0]
        mfcc = np.pad(mfcc, ((0, pad_width), (0, 0)), mode="constant")
    else:
        mfcc = mfcc[:max_len, :]

    return mfcc

In [6]:
def split_audio(audio, sr, win_sec=3, overlap_ratio=0.2):
    win_len = int(win_sec * sr)
    hop_len = int(win_len * (1-overlap_ratio))
    segments = []

    for start in range(0, len(audio)-win_len, hop_len):
        segments.append(audio[start:start+win_len])

    return segments


In [7]:
class_folders = {
    "Asthma": asthma_data,
    "COPD": copd_data,
    "Healthy": healthy_data
}

all_files = []
for label, folder in class_folders.items():
    for f in os.listdir(folder):
        if f.endswith('.wav'):
            all_files.append((os.path.join(folder, f), label))

train_files, test_files = train_test_split(
    all_files, test_size=0.2, stratify=[x[1] for x in all_files], random_state=42
)

In [8]:
def prepare_data(file_list, augment=False):
    X, y = [], []

    for path, label in file_list:
        try:
            audio, sr = librosa.load(path, sr=22050)
            segments = split_audio(audio, sr)

            for seg in segments:
                feat = extract_mfcc(seg, sr)
                X.append(feat)
                y.append(label)

                if augment:
                    aug = apply_augmentations(seg)
                    X.append(extract_mfcc(aug, sr))
                    y.append(label)

        except Exception as e:
            print(f"Hata: {path} -> {e}")

    return np.array(X), np.array(y)



In [9]:
print("Veriler işleniyor...")
X_train_raw, y_train_raw = prepare_data(train_files, augment=True)
X_test_raw, y_test_raw = prepare_data(test_files, augment=False)

Veriler işleniyor...


In [10]:
# Padding
X_train_pad = pad_sequences(X_train_raw, padding="post", dtype="float32")
X_test_pad = pad_sequences(X_test_raw, padding="post", dtype="float32", 
                            maxlen=X_train_pad.shape[1])

# Channel ekle
X_train_cnn = X_train_pad
X_test_cnn  = X_test_pad

# Label Encoding
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_raw)
y_test_enc  = le.transform(y_test_raw)


In [11]:
def build_cnn_1d(input_shape, layers):
    model = Sequential()

    for i, f in enumerate(layers):
        if i == 0:
            model.add(
                Conv1D(
                    filters=f,
                    kernel_size=3,
                    padding="same",
                    kernel_regularizer=l2(0.001),
                    input_shape=input_shape
                )
            )
        else:
            model.add(
                Conv1D(
                    filters=f,
                    kernel_size=3,
                    padding="same",
                    kernel_regularizer=l2(0.001)
                )
            )

        model.add(BatchNormalization())
        model.add(Activation('relu'))
        model.add(MaxPooling1D(2))
        model.add(Dropout(0.2))

    model.add(Flatten())
    model.add(Dense(128, activation="relu"))
    model.add(Dropout(0.3))

    model.add(Dense(3, activation="softmax"))

    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


In [12]:


configs = {

    "1_layer": [[32],[64],[128],[256],[512]],

    "2_layer": [[32,64],[64,128],[128,256],[256,512]],

    "3_layer": [[32,64,128],[64,128,256],[128,256,512]]

}

In [13]:
results = []

for group, cfgs in configs.items():
    print(f"\n--- {group} Test Ediliyor ---")

    for filters in cfgs:
        model = build_cnn_1d(X_train_cnn.shape[1:], filters)

        model.fit(
            X_train_cnn, y_train_enc,
            validation_data=(X_test_cnn, y_test_enc),
            epochs=100,
            batch_size=32,
            verbose=0
        )

        loss, acc = model.evaluate(X_test_cnn, y_test_enc, verbose=0)
        print(f"Filters {filters} -> Acc: {acc:.4f}")

        results.append({"cfg": filters, "acc": acc})



--- 1_layer Test Ediliyor ---


c:\Users\MONSTER\Desktop\LungSoundAnlysis\LungSoundAnlysis\venv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Filters [32] -> Acc: 0.5817
Filters [64] -> Acc: 0.5992
Filters [128] -> Acc: 0.5584
Filters [256] -> Acc: 0.5603
Filters [512] -> Acc: 0.5136

--- 2_layer Test Ediliyor ---
Filters [32, 64] -> Acc: 0.6401
Filters [64, 128] -> Acc: 0.6440
Filters [128, 256] -> Acc: 0.6284
Filters [256, 512] -> Acc: 0.6556

--- 3_layer Test Ediliyor ---
Filters [32, 64, 128] -> Acc: 0.6342
Filters [64, 128, 256] -> Acc: 0.5934
Filters [128, 256, 512] -> Acc: 0.5973


In [14]:
best = max(results, key=lambda x: x["acc"])
print("\n🔥 EN İYİ MODEL 🔥")
print(best)



🔥 EN İYİ MODEL 🔥
{'cfg': [256, 512], 'acc': 0.655642032623291}
